<a href="https://colab.research.google.com/github/kate7094-tech/mytest/blob/main/02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Library import

2. Data download

3. EDA

4. Duplicate 처리

5. RDKit Descriptor 생성

6. RDKit Descriptor 전처리
    (desc_raw
    desc_imp
    desc_const
    desc_final)

7. Morgan Fingerprint

8. Machine Learning (RandomForest, SVR, XGboost, DNN)

9. SHAP

# 1. 데이터셋
Library import

Data download

EDA

Duplicate 처리

In [1]:
!pip install PyTDC
!pip install rdkit
!pip install mordred
!pip install xgboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pytdc to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.3/151.3 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 48.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 17.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.2/151.2 kB 15.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... d

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.8/128.8 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.0 MB/s eta 0:00:00
  Created wheel for mordred: filename=mordred-1.2.0-py3-none-any.whl size=176718 sha256=6989906a6c16d0e5655663cdb923284c49c0d5206f64b23072a757d1d83c28b1
  Stored in directory: /root/.cache/pip/wheels/e8/79/b8/f4f1dfbb736c2b8605cf5068cd633f4d2869defb89908aef93
Successfully built mordred
  Attempting uninstall: networkx
    Found existing installation: networkx 3.6.1
    Uninstalling networkx-3.6.1:
      Successfully uninstalled networkx-3.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
mapclassify 2.10.0 requires scikit-learn>=1.4, but you have scikit-learn 1.2.2 which 

In [2]:
import numpy as np
import pandas as pd
import sklearn
from rdkit import Chem
from tdc.single_pred import ADME

print("numpy:", np.__version__)
print("sklearn:", sklearn.__version__)
print("import 성공")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
data = ADME(name="Caco2_Wang")
df = data.get_data()

df.head()

In [ ]:
df.info()
df.describe()
df.head()
df.shape
df["Y"].describe()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df["Y"], bins=30, edgecolor="black")
plt.xlabel("Caco-2 permeability (Y)")
plt.ylabel("Count")
plt.title("Distribution of Caco-2 permeability")
plt.show()

In [ ]:
splits = data.get_split()

print(splits.keys())

for name, split in splits.items():
    print(name, split.shape)

print(splits["train"].head())
print(splits["valid"].head())
print(splits["test"].head())

In [ ]:
df["Drug"].duplicated().sum()
dup = df[df["Drug"].duplicated(keep=False)]

dup

In [ ]:
df_raw = df.copy()

# Ketotifen 제거
ketotifen_smiles = "CN1CCC(=C2c3ccccc3CC(=O)c3sccc32)CC1"
df = df[df["Drug"] != ketotifen_smiles]
df.shape

In [ ]:
df[df["Drug"].duplicated(keep=False)].sort_values("Drug")

In [ ]:
# 나머지 중복은 평균내기
df = (
    df
    .groupby("Drug", as_index=False)
    .agg({
        "Drug_ID": "first",
        "Y": "mean"
    })
)

print(df.shape)
print(df["Drug"].duplicated().sum())

In [ ]:
df.isnull().sum()

# 2. 분자 표현

RDKit Descriptor

Morgan Fingerprint

RDKit Descriptor + Morgan Fingerprint

In [ ]:
# 1. RDKit Descriptor 생성
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

def calculate_rdkit_descriptors(smiles):
    """SMILES → RDKit Descriptor 계산"""
    mol = Chem.MolFromSmiles(smiles)

    descriptors = {}
    for name, func in Descriptors._descList:
        descriptors[name] = func(mol)

    return descriptors

# 모든 화합물에 대해 Descriptor 계산
desc = df["Drug"].apply(calculate_rdkit_descriptors)

# DataFrame으로 변환
desc_df = pd.DataFrame(desc.tolist())

print(desc_df.shape)

In [ ]:
# 결측치 확인
missing = desc_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print("Descriptors with missing values:", len(missing))
print("Total missing values:", missing.sum())
display(missing)

# 무한대 확인
inf = np.isinf(desc_df).sum()
inf = inf[inf > 0]

print("Descriptors with infinite values:", len(inf))
print("Total infinite values:", inf.sum())
display(inf)

In [ ]:
# RDKit descriptor 원본
desc_raw = pd.DataFrame(desc.tolist())

In [ ]:
# 3. 결측치 평균값으로 대체

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")

desc_imp = pd.DataFrame(
    imputer.fit_transform(desc_raw),
    columns=desc_raw.columns
)

# 결측치가 모두 없어졌는지 확인
print(desc_df.isnull().sum().sum())

In [ ]:
# 4. 상수(Constant) Descriptor 제거

constant_features = desc_imp.columns[desc_imp.nunique() <= 1]
desc_const = desc_imp.drop(columns=constant_features)

In [ ]:
# 5. 높은 상관관계 Descriptor 제거

# 절댓값 상관계수 계산
corr_matrix = desc_const.corr().abs()

# 중복 계산을 피하기 위해 상삼각행렬만 사용
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

# 상관계수 0.95 이상인 Descriptor 제거 후보
to_drop = [
    column
    for column in upper.columns
    if any(upper[column] > 0.95)
]

# 제거
desc_final = desc_const.drop(columns=to_drop)

# 6. 단계별 결과 확인
print("raw:", desc_raw.shape)
print("imputed:", desc_imp.shape)
print("constant removed:", desc_const.shape)
print("high corr removed:", desc_final.shape)
print("constant features:", len(constant_features))
print("high corr features:", len(to_drop))

In [ ]:
# Morgan Fingerprint
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.DataStructs import ConvertToNumpyArray

In [ ]:
# 1. Morgan Fingerprint 생성 함수
# radius=2 → ECFP4
# nBits=1024 → 1024-bit Fingerprint

def calculate_morgan_fp(smiles, radius=2, nBits=1024):

    # SMILES → Mol 객체
    mol = Chem.MolFromSmiles(smiles)

    # Morgan Fingerprint 생성
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=radius,
        nBits=nBits
    )

    # RDKit 객체 → NumPy 배열 변환
    arr = np.zeros((nBits,), dtype=int)
    ConvertToNumpyArray(fp, arr)

    return arr

# 2. 전체 데이터에 적용

morgan = df["Drug"].apply(calculate_morgan_fp)

# DataFrame으로 변환
morgan_df = pd.DataFrame(
    morgan.tolist(),
    columns=[f"FP_{i}" for i in range(1024)]
)

print(morgan_df.shape)

# 첫 5개 화합물 확인
morgan_df.head()

In [ ]:
# 3. 전처리
# 결측치 확인
print(morgan_df.isnull().sum().sum())

# 상수 Bit 확인
constant_fp = morgan_df.columns[
    morgan_df.nunique() <= 1
]
print(f"Constant fingerprints: {len(constant_fp)}")

# 상수 Bit 제거
morgan_const = morgan_df.drop(columns=constant_fp)

print("Original Morgan FP:", morgan_df.shape)
print("After removing constant bits:", morgan_const.shape)
print("Constant bits removed:", len(constant_fp))

# 희귀 bit 제거
# threshold=0.01은 분산이 너무 낮은 bit 제거
# 즉, 대부분 0이거나 대부분 1인 bit를 제거
from sklearn.feature_selection import VarianceThreshold
selector = VarianceThreshold(threshold=0.01)

morgan_var_array = selector.fit_transform(morgan_const)

# 남은 column 이름 추출
selected_columns = morgan_const.columns[selector.get_support()]

morgan_final = pd.DataFrame(
    morgan_var_array,
    columns=selected_columns
)

print("After removing low-variance bits:", morgan_final.shape)

In [ ]:
# Descriptor + Morgan 결합
X_combined = pd.concat(
    [desc_final, morgan_final],
    axis=1
)

print(X_combined.shape)

In [ ]:
# Final Input Matrices

X_desc = desc_final.copy()
X_morgan = morgan_final.copy()

X_combined = pd.concat(
    [desc_final.reset_index(drop=True),
     morgan_final.reset_index(drop=True)],
    axis=1
)

y = df["Y"].reset_index(drop=True)

df = df.reset_index(drop=True)
X_desc = X_desc.reset_index(drop=True)
X_morgan = X_morgan.reset_index(drop=True)
X_combined = X_combined.reset_index(drop=True)

print("X_desc:", X_desc.shape)
print("X_morgan:", X_morgan.shape)
print("X_combined:", X_combined.shape)
print("y:", y.shape)

# 3. ML

In [ ]:
# 기존 TDC split (70 10 20)
print(splits["train"].shape)
print(splits["valid"].shape)
print(splits["test"].shape)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# 전체 index 생성
idx = np.arange(len(df))

# 먼저 test 20% 분리
train_valid_idx, test_idx = train_test_split(
    idx,
    test_size=0.2,
    random_state=42
)

# 남은 80% 중에서 valid를 1/8로 분리
# 전체 기준으로 0.8 × 0.125 = 0.1 → valid 10%
train_idx, valid_idx = train_test_split(
    train_valid_idx,
    test_size=0.125,
    random_state=42
)

# RDKit descriptor
X_desc_train = X_desc.iloc[train_idx]
X_desc_valid = X_desc.iloc[valid_idx]
X_desc_test  = X_desc.iloc[test_idx]

# Morgan fingerprint
X_morgan_train = X_morgan.iloc[train_idx]
X_morgan_valid = X_morgan.iloc[valid_idx]
X_morgan_test  = X_morgan.iloc[test_idx]

# Combined
X_combined_train = X_combined.iloc[train_idx]
X_combined_valid = X_combined.iloc[valid_idx]
X_combined_test  = X_combined.iloc[test_idx]

# Target
y_train = y.iloc[train_idx]
y_valid = y.iloc[valid_idx]
y_test  = y.iloc[test_idx]

print("train:", len(train_idx))
print("valid:", len(valid_idx))
print("test:", len(test_idx))

## 3-1. Random Forest Baseline

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd


# 성능 평가 함수
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }


# Random Forest 모델
rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

In [ ]:
# RDKit Descriptor로 학습

rf.fit(X_desc_train, y_train)

y_pred_desc = rf.predict(X_desc_test)

result_desc = evaluate_model(y_test, y_pred_desc)

result_desc

In [ ]:
# Morgan Fingerprint로 학습

rf.fit(X_morgan_train, y_train)

y_pred_morgan = rf.predict(X_morgan_test)

result_morgan = evaluate_model(y_test, y_pred_morgan)

result_morgan

In [ ]:
# RDKit + Morgan으로 학습

rf.fit(X_combined_train, y_train)

y_pred_combined = rf.predict(X_combined_test)

result_combined = evaluate_model(y_test, y_pred_combined)

result_combined

In [ ]:
# 결과 표로 정리

rf_results = pd.DataFrame([
    {"Representation": "RDKit Descriptor", **result_desc},
    {"Representation": "Morgan FP", **result_morgan},
    {"Representation": "RDKit + Morgan", **result_combined}
])

rf_results

## 3-2. SVR Baseline

In [ ]:
# SVR Baseline
# RDKit만 Scaling, Morgan은 Binary 그대로 사용

from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# 1. SVR 모델 생성
# RDKit descriptor용: scaling + SVR

svr_desc = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf", C=1.0, epsilon=0.1))
])

# Morgan fingerprint용: scaling 없이 SVR
svr_morgan = SVR(
    kernel="rbf",
    C=1.0,
    epsilon=0.1
)

# 2. Combined 데이터 생성
# RDKit 부분만 scaling 후 Morgan과 결합

scaler = StandardScaler()

# train 데이터로만 scaler 학습
X_desc_train_scaled = scaler.fit_transform(X_desc_train)

# valid/test는 train에서 학습한 scaler로 변환만 수행
X_desc_valid_scaled = scaler.transform(X_desc_valid)
X_desc_test_scaled = scaler.transform(X_desc_test)

# Morgan은 0/1 binary 그대로 사용
X_combined_train_mixed = np.concatenate(
    [X_desc_train_scaled, X_morgan_train.values],
    axis=1
)

X_combined_valid_mixed = np.concatenate(
    [X_desc_valid_scaled, X_morgan_valid.values],
    axis=1
)

X_combined_test_mixed = np.concatenate(
    [X_desc_test_scaled, X_morgan_test.values],
    axis=1
)

# Combined용 SVR
svr_combined = SVR(
    kernel="rbf",
    C=1.0,
    epsilon=0.1
)

In [ ]:
# 3. RDKit Descriptor 학습 및 평가

svr_desc.fit(X_desc_train, y_train)
y_pred_desc = svr_desc.predict(X_desc_test)
result_desc = evaluate_model(y_test, y_pred_desc)

# 4. Morgan Fingerprint 학습 및 평가

svr_morgan.fit(X_morgan_train, y_train)
y_pred_morgan = svr_morgan.predict(X_morgan_test)
result_morgan = evaluate_model(y_test, y_pred_morgan)

# 5. RDKit + Morgan 학습 및 평가

svr_combined.fit(X_combined_train_mixed, y_train)
y_pred_combined = svr_combined.predict(X_combined_test_mixed)
result_combined = evaluate_model(y_test, y_pred_combined)

# 6. 결과 정리

svr_results_mixed = pd.DataFrame([
    {"Model": "SVR", "Representation": "RDKit Descriptor_scaled", **result_desc},
    {"Model": "SVR", "Representation": "Morgan FP_no scaling", **result_morgan},
    {"Model": "SVR", "Representation": "RDKit_scaled + Morgan", **result_combined}
])

svr_results_mixed

## 3-3. XGBoost

In [ ]:
from xgboost import XGBRegressor
import pandas as pd

xgb = XGBRegressor(
    objective="reg:squarederror"
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

In [ ]:
# 1. RDKit Descriptor
xgb.fit(X_desc_train, y_train)
y_pred_desc = xgb.predict(X_desc_test)
result_desc = evaluate_model(y_test, y_pred_desc)

# 2. Morgan Fingerprint
xgb.fit(X_morgan_train, y_train)
y_pred_morgan = xgb.predict(X_morgan_test)
result_morgan = evaluate_model(y_test, y_pred_morgan)

# 3. RDKit + Morgan
xgb.fit(X_combined_train, y_train)
y_pred_combined = xgb.predict(X_combined_test)
result_combined = evaluate_model(y_test, y_pred_combined)

# 4. 결과 정리
xgb_results = pd.DataFrame([
    {"Model": "XGBoost", "Representation": "RDKit Descriptor", **result_desc},
    {"Model": "XGBoost", "Representation": "Morgan FP", **result_morgan},
    {"Model": "XGBoost", "Representation": "RDKit + Morgan", **result_combined}
])

xgb_results